# 2.3.1 — Word count distribuito sul full text di CORD-19

Questo notebook **importa** `word_count.py`: non riscrive l'algoritmo e non lancia
sottoprocessi. Tutta la logica sta nel modulo, qui c'è l'esecuzione e la lettura dei
risultati, così esiste una sola sorgente di verità.

L'algoritmo è quello dell'assignment (§2.3.1):

| fase | cosa produce |
|---|---|
| **Map** | per ogni documento *D*, le coppie `(w, cp(w))` — quante volte la parola `w` compare in *D* |
| **Reduce** | per ogni parola `w`, `c(w) = Σ cp(w)` su tutti i documenti |

Struttura dati: **Bag**, come raccomanda il testo («we recommend utilizing the RDD/Bag
data structure»).

Input: `data/silver/paragraphs` — una riga per paragrafo, già sanificato dalla pipeline
di conversione (vedi `DATA_DICTIONARY.md`).

Le scelte di pulizia del testo sono tutte motivate da misure sul corpus: la
giustificazione riga per riga è nel README di questa cartella.

## 1 · Cluster

Dove gira il calcolo lo decide `cluster.txt` alla root del repo (git-ignored), non il
codice: sul Mac parte un `LocalCluster`, sulla VM un `SSHCluster` sui nodi elencati.
Lo stesso notebook gira nei due posti senza modifiche.

In [ ]:
import sys
import time
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "Giulia" else Path.cwd()
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / "Giulia"))

from cluster import get_client
import word_count as wc

client, cluster = get_client(repo_root=REPO)
client

## 2 · I dati

**Una partizione è un gruppo di file Parquet, e il worker apre i suoi.** Non
`dd.read_parquet(...).to_bag()`, che sarebbe una riga: attraverso quello il numero di
partizioni non è una manopola che imposti, è una proprietà che scopri — e dipende dalla
forma della query. Misurato su questo corpus: leggere tre colonne con un filtro booleano
in mezzo dava **990** partizioni, la lettura piana a due colonne ne dà **1979**, a parità
di 1979 file.

Il benchmark obbligatorio ha il numero di partizioni come **variabile indipendente**:
un numero deciso dall'ottimizzatore lo squalifica. Raggruppando i file si ottiene
esattamente `k`, senza rimescolamenti da pagare — e produzione e benchmark diventano lo
stesso identico codice.

In [ ]:
INPUT = REPO / "data" / "silver" / "paragraphs"

files = wc.paragraph_files(INPUT)                       # i .parquet, in ordine numerico
paragraphs = wc.read_groups(wc.split_evenly(files, len(files)))   # una partizione per file
print(f"{len(files)} file  ->  {paragraphs.npartitions} partizioni")
paragraphs.take(1)

## 3 · Come si comporta la sanitizzazione

Un controllo a occhio prima di lanciare il calcolo vero: cosa resta di un testo con
trattini tipografici, lettere greche e stop-word.

In [ ]:
demo = "The SARS–CoV-2 virus and TNF-α were measured at 5 µg/mL in Müller's study."
print(wc.sanitize(demo))
print(wc.words(demo))

## 4 · Il grafo delle due fasi

`word_count` restituisce i due Bag, ancora **lazy**: nulla è stato calcolato.

**La fase Map non fa passare niente in rete.** `doc_counts` ha lo stesso numero di
partizioni dell'input: il conteggio per documento avviene *dentro* la partizione, ed è il
*combiner* del MapReduce classico. Ciò che esce dal worker è una entry per
`(documento, parola)`, non una per occorrenza.

**La fase Reduce ha due formulazioni**, che calcolano la stessa identica cosa
(verificato sul corpus intero: 6.037.808 parole, 785.753.529 occorrenze, ogni conteggio
uguale) ma si comportano in modo molto diverso:

| | `split_out=0` — `Bag.foldby` | `split_out=16` — groupby DataFrame |
|---|---|---|
| partizioni in uscita | **1** | 16 |
| coda del calcolo | un task solo, seriale | 16 task in parallelo |
| memoria del task finale | tutto il vocabolario (~1,5–2 GB) | ~vocabolario/16 |

`Bag.foldby` e `Bag.frequencies` riducono **sempre** a una partizione sola: tutto lo
spazio delle chiavi deve stare in un singolo task, su un singolo worker. Con la parola
come chiave è fattibile — il vocabolario satura al crescere del corpus (legge di
Heaps) — ma resta una coda seriale. Misurato sul corpus intero:

```
7,0 GB/worker,  foldby         1128 s,  0 worker uccisi
3,4 GB/worker,  foldby         1762 s,  3 worker uccisi
3,4 GB/worker,  split_out=16    274 s,  0 worker uccisi
```

Da qui il default. Il testo dell'assignment permette esplicitamente di passare da Bag a
DataFrame, e la fase Map — dove sta il lavoro vero — resta Bag.

In [ ]:
doc_counts, global_counts = wc.word_count(paragraphs)
print("input        partitions:", paragraphs.npartitions)
print("doc_counts   partitions:", doc_counts.npartitions, "  (Map: nessuno shuffle)")
print("global_counts partitions:", global_counts.npartitions)

_, foldby_counts = wc.word_count(paragraphs, split_out=0)
print("con split_out=0        :", foldby_counts.npartitions, "  (foldby: coda seriale)")

## 5 · Smoke test

Prima del run completo, le stesse identiche operazioni su poche partizioni.

In [ ]:
smoke = wc.read_groups(wc.split_evenly(files[:8], 8))
_, smoke_counts = wc.word_count(smoke)
smoke_counts.topk(10, key=1).compute()

## 6 · Run completo

`topk` pota presto: ogni partizione inoltra solo le sue prime N, quindi non serve
materializzare tutto il vocabolario per avere la classifica.

In [ ]:
TOP_N = 20

started = time.perf_counter()
top = global_counts.topk(TOP_N, key=1).compute()
elapsed = time.perf_counter() - started
print(f"{elapsed:.1f} s")

for word, count in top:
    print(f"{count:>12,}  {word}")

## 7 · Verifica dell'invariante

Il Reduce **raggruppa e basta**: non perde né inventa occorrenze. Il controllo somma i
conteggi prima e dopo e verifica che coincidano.

Nessun totale assoluto scritto a mano: il dump locale e il corpus della VM sono dataset
diversi, quindi l'unica garanzia sensata è strutturale (`PROJECT_CONTEXT.md`, regola 8.2).

Costa: sommare i conteggi per-documento obbliga a percorrere tutta la tabella
intermedia, mentre `topk` può potare. Si lancia in sviluppo e prima di una consegna,
non a ogni esecuzione.

In [ ]:
import dask

after_map, after_reduce = dask.compute(doc_counts.pluck(1).sum(), global_counts.pluck(1).sum())
assert after_map == after_reduce, (after_map, after_reduce)
print(f"invariante ok: {after_map:,} occorrenze prima e dopo il reduce")

## 8 · Barplot

Il grafico che l'assignment chiede esplicitamente («create a barplot of the top
words»).

In [ ]:
OUT = Path("~/mapd-out/word_count").expanduser()
OUT.mkdir(parents=True, exist_ok=True)

wc.barplot(top, OUT / "top_words.png", f"Top {len(top)} words in the CORD-19 body text")

from IPython.display import Image
Image(str(OUT / "top_words.png"))

## 9 · Benchmark obbligatori

Le linee guida del corso li richiedono esplicitamente — tempo di esecuzione contro
**numero di partizioni** e contro **numero di worker** — e senza, il progetto è
considerato incompleto. Sono due. Non ci sono altre curve obbligatorie.

**Qui non si misura niente: si legge e si disegna.** La divisione è netta e voluta:

| | dove | cosa fa |
|---|---|---|
| misurare | `Giulia/bench_word_count.py`, sul cluster | scrive un CSV, una riga per ripetizione |
| capire | questo notebook | legge il CSV e disegna |

Serve a due cose concrete: la campagna dura ore e non deve dipendere da un notebook
aperto, e per rifare un grafico non si rioccupa il cluster.

```bash
tmux new -s bench
source ~/pyvenv/bin/activate && cd ~/MAPD-Project
python Giulia/bench_word_count.py ~/mapd-data/silver/paragraphs 2>&1 | tee ~/bench.log
```

Un comando solo. La campagna fa, in ordine: le quattro fotografie (§9.3), la curva sulle
partizioni, la curva sui worker. Il lavoro cronometrato è **quello che si consegna** —
Map, Reduce e scrittura del vocabolario — non una versione più comoda da misurare.

**Il numero di worker non si cambia rimpicciolendo il cluster.** Su `SSHCluster` si può
solo scendere, e questo obbligherebbe a ricordarsi un ordine di esecuzione: per ogni punto
si accende invece un cluster nuovo con i primi *N* host di `cluster.txt`. Costa un minuto
a punto, e ogni misura parte da uno stato pulito.

In [ ]:
import pandas as pd

MISURE = Path("~/mapd-out/bench/misure.csv").expanduser()
misure = pd.read_csv(MISURE) if MISURE.exists() else pd.DataFrame()

# Le righe senza `secondi` non sono buchi: sono configurazioni che NON HANNO COMPLETATO,
# ed è un risultato. Restano nel CSV e si contano; spariscono solo dai grafici.
if misure.empty:
    print(f"nessuna misura in {MISURE} — la campagna non è ancora stata lanciata")
else:
    for curva, gruppo in misure.groupby("curva"):
        fallite = gruppo["secondi"].isna().sum()
        print(f"{curva:<12} {len(gruppo):>3} misure, {fallite} non completate, "
              f"{gruppo['valore'].nunique()} punti")

### 9.1 · Tempo vs numero di partizioni *(obbligatorio)*

**Gli stessi identici dati, tagliati in modo diverso.** Cluster fisso, `split_out` fisso:
si muove solo `k`. A dato fisso «numero di partizioni» *è* «dimensione di una partizione».

Cosa aspettarsi, ed è meglio saperlo prima di guardare:

- **a sinistra il degrado è vero e può diventare fallimento.** Con `k=4` sul corpus intero
  una partizione è ~2,5 GB di testo in un singolo task, su worker da 4 GB. Un punto che non
  completa è una riga della tabella, non un buco;
- **a destra la curva resta piatta, non risale.** Il ramo «troppe partizioni» si alza quando
  i task diventano più corti del tempo di distribuirli: qui un task tokenizza ~5 MB di
  testo, cioè secondi. Per farlo salire servirebbero decine di migliaia di partizioni, e
  più fine di un file per partizione non si può andare (un Parquet del silver ha un solo
  row-group). **Va scritto nella relazione**: una curva spiegata vale più di una bella.

In [ ]:
import matplotlib.pyplot as plt

def curva(frame, x_label, titolo, ax):
    """Media delle ripetizioni, con la dispersione (min-max) come barra d'errore.

    La dispersione non è decorazione: se due configurazioni distano meno di quanto
    oscillano le loro ripetizioni, quella differenza non esiste."""
    gruppi = frame.dropna(subset=["secondi"]).groupby("valore")["secondi"]
    x = sorted(gruppi.groups)
    medie = gruppi.mean().loc[x]
    basso = medie - gruppi.min().loc[x]
    alto = gruppi.max().loc[x] - medie
    ax.errorbar(x, medie, yerr=[basso, alto], marker="o", capsize=4, color="#2f6f73")
    ax.set_xlabel(x_label); ax.set_ylabel("secondi"); ax.set_title(titolo)
    ax.grid(alpha=0.25); ax.set_ylim(bottom=0)
    return medie

if not misure.empty:
    partizioni = misure[misure.curva == "partizioni"]
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.set_xscale("log")
    medie = curva(partizioni, "numero di partizioni", "Word count: tempo vs partizioni", ax)
    plt.show()

    display(partizioni.groupby("valore")[["secondi"]].agg(["mean", "min", "max", "count"]).round(1))
    non_completate = partizioni[partizioni.secondi.isna()]
    if not non_completate.empty:
        print("non hanno completato:")
        display(non_completate[["valore", "errore"]])

### 9.2 · Tempo vs numero di worker *(obbligatorio)*

**Stessi dati, stesso taglio: cambia solo quanta macchina lavora.** I dati sono replicati
sul disco di ogni VM, quindi ogni worker legge sempre da casa propria: togliere worker non
sposta un collo di bottiglia sulla rete, e la curva parla di calcolo. (Con l'architettura
NFS precedente non sarebbe stato vero.)

Le due grandezze che il corso chiede davvero dietro «tempo vs numero di esecutori»:

- **speedup** *S(n) = T(1)/T(n)* — quante volte va più veloce di un worker solo;
- **efficienza** *E(n) = S(n)/n* — quanta parte di ogni worker aggiunto sta davvero
  producendo. Vale 1 se il guadagno è perfetto.

L'efficienza è quella onesta: cala sempre, e **quanto** cala misura la parte non
parallela del lavoro — cioè, qui, la coda del Reduce.

In [ ]:
if not misure.empty:
    worker = misure[misure.curva == "worker"]
    fig, (sopra, sotto) = plt.subplots(2, 1, figsize=(7, 7.5), sharex=True)
    medie = curva(worker, "", "Word count: tempo vs numero di worker", sopra)

    riferimento = medie.index.min()          # il punto più piccolo che ha completato
    speedup = medie[riferimento] / medie
    ideale = medie.index / riferimento
    sotto.plot(medie.index, speedup, marker="o", color="#2f6f73", label="speedup misurato")
    sotto.plot(medie.index, ideale, "--", color="#999", label="speedup ideale")
    sotto.plot(medie.index, speedup / ideale, marker="s", color="#b4674d", label="efficienza")
    sotto.set_xlabel("numero di worker"); sotto.grid(alpha=0.25)
    sotto.set_ylim(bottom=0); sotto.legend()
    plt.show()

    display(pd.DataFrame({"secondi": medie.round(1),
                          "speedup": speedup.round(2),
                          "efficienza": (speedup / ideale).round(2)}))

### 9.3 · Le quattro fotografie

Oltre ai numeri, la campagna produce quattro pagine HTML con `performance_report`, che
costa **due righe** e dà quello che nessuna curva può dare: la **linea del tempo di ogni
task su ogni worker** (scheda *Task Stream*) e la **memoria di ogni worker nel tempo**
(scheda *System*).

| file | condizione | cosa ci si vede |
|---|---|---|
| `report_riferimento.html` | corpus intero, `split_out=16` | il lavoro che si consegna |
| `report_split_out.html` | fetta, `split_out=16` | il termine di paragone |
| `report_foldby.html` | fetta, `split_out=0` | **la coda seriale**: un task solo, gli altri worker bianchi |
| `report_un_worker.html` | fetta, un worker | il cluster mezzo vuoto |

Il `foldby` gira su una **fetta** e non sul corpus intero per un motivo misurato: sul
corpus intero è già morto una volta, con `KilledWorker` sul task finale. Quello che il
report deve far vedere è **strutturale** e si vede identico a qualsiasi scala: non si
rischia una notte per una fotografia.

I file si aprono nel browser. Sono autoconsistenti (`mode="inline"`): senza quel
parametro l'HTML scarica BokehJS da `cdn.bokeh.org` e resta **bianco** appena lo si guarda
senza internet — cioè, tipicamente, dopo averlo copiato giù dal cluster.

## 10 · Dove finisce il Reduce

Non è fra i benchmark obbligatori ed è il risultato più istruttivo del task. Non ha avuto
bisogno di nessuna impalcatura: sono **due lanci di `word_count.py` con un flag diverso**,
sul corpus intero, e i numeri scritti a mano.

| configurazione | tempo | worker uccisi | risultato |
|---|---:|---:|---|
| 7,0 GB per worker, `foldby` | 1.128 s | 0 | 6.037.808 parole |
| 3,4 GB per worker, `foldby` | 1.762 s | **3** | identico |
| 3,4 GB per worker, `split_out=16` | **274 s** | 0 | identico |

**4,1× più veloce del miglior run con `foldby`, su worker grandi la metà.** Stesso
risultato, chiave per chiave.

La lettura giusta non è «servivano più GB»: `Bag.foldby` restituisce **una** partizione
per costruzione — è l'ultimo argomento nel sorgente di dask, `dask/bag/core.py:1418` —
quindi un solo task macina un dizionario Python da 6 milioni di voci mentre gli altri
worker stanno fermi. Con `split_out` diventano 16 task in parallelo, ognuno con circa un
sedicesimo del vocabolario, in colonne Arrow invece che in oggetti Python. La memoria era
il sintomo; il collo di bottiglia era il parallelismo.

**Morale che vale per tutti e quattro i task:** un oggetto grosso dev'essere il *risultato
di tanti task piccoli*, mai la *variabile locale di un task grosso* — perché solo il primo
Dask lo sa gestire, spostare e riversare su disco.

## 11 · Chiusura

In [ ]:
client.close()
if cluster is not None:
    cluster.close()
print("cluster chiuso")